In [2]:
import os
import pandas as pd

original_csv_folder = "/mnt/sdb/yuran/av_hubert/datasets/multivsr/original_metadata/"
target_csv_folder = "/mnt/sdb/yuran/av_hubert/datasets/multivsr/"

csv_names = ["train_major.csv", "val_major.csv"]

youtube_ids_all = set()

for name in csv_names:
    original_path = os.path.join(original_csv_folder, name)
    target_path = os.path.join(target_csv_folder, name)
    
    df = pd.read_csv(original_path)
    df_en = df[df['language'] == 'en']
    
    # extract youtube/video ids from file_path (before first '/')
    df_en['youtube_id'] = df_en['file_path'].str.split('/', n=1).str[0]
    video_ids = df_en['youtube_id'].unique()
    
    frac=30/440
    if name.startswith("val"):
        frac=0.5*30/440
    sampled_video_ids = pd.Series(video_ids).sample(frac=frac, random_state=42).tolist() 
    
    # sum the "length" of selected videos
    total_length = df_en[df_en['youtube_id'].isin(sampled_video_ids)]['length'].sum()
    print(f"Total length of sampled videos for {name}: {total_length/25/3600} hours") # fps=25
    
    # keep only rows corresponding to sampled videos
    df_sampled = df_en[df_en['youtube_id'].isin(sampled_video_ids)].reset_index(drop=True)
    
    df_sampled.to_csv(target_path, index=False)
    print(f"Processed {name}: kept {len(df_sampled)} out of {len(df_en)} English entries for split {name.split('_')[0]}")
    
    # write this split's unique youtube ids
    ytids_path = os.path.join(target_csv_folder, f"ytids_{name.split('_')[0]}.txt")
    with open(ytids_path, 'w') as f:
        for yt_id in sorted(sampled_video_ids):
            f.write(f"{yt_id}\n")
    
    youtube_ids_all.update(sampled_video_ids)

# write all unique youtube ids across splits
ytids_path = os.path.join(target_csv_folder, "ytids.txt")
with open(ytids_path, 'w') as f:
    for yt_id in sorted(youtube_ids_all):
        f.write(f"{yt_id}\n")

print(f"Wrote {len(youtube_ids_all)} unique YouTube IDs to {ytids_path}")


/tmp/ipykernel_1479779/1255424626.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_en['youtube_id'] = df_en['file_path'].str.split('/', n=1).str[0]


Total length of sampled videos for train_major.csv: 299.8376222222222 hours
Processed train_major.csv: kept 321340 out of 4758506 English entries for split train
Total length of sampled videos for val_major.csv: 2.7819000000000003 hours
Processed val_major.csv: kept 3037 out of 75378 English entries for split val
Wrote 4246 unique YouTube IDs to /mnt/sdb/yuran/av_hubert/datasets/multivsr/ytids.txt


/tmp/ipykernel_1479779/1255424626.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_en['youtube_id'] = df_en['file_path'].str.split('/', n=1).str[0]


In [ ]:
# import os
# import pandas as pd

# # paths
# base_folder = "/mnt/sdb/yuran/av_hubert/datasets/multivsr"
# input_csv_folder = base_folder
# input_videos_folder = os.path.join(base_folder, "videos")
# output_folder = os.path.join(base_folder, "final_metadata")

# os.makedirs(output_folder, exist_ok=True)

# csv_names = ["train_major.csv", "val_major.csv"]
# FPS = 25

# # ------------------------------------------------
# # 1. collect existing video IDs from .mp4 files
# # ------------------------------------------------



# existing_video_ids = {
#     os.path.splitext(f)[0]
#     for f in os.listdir(input_videos_folder)
#     if f.endswith(".mp4")
# }

# # remove videos in input_videos_folder, but not in /mnt/sdb/yuran/av_hubert/datasets/multivsr/ytids.txt
# with open(os.path.join(base_folder, "ytids.txt"), "r") as f:
#     valid_ytids = {line.strip() for line in f}
#     for existing_video_id in list(existing_video_ids):
#         if existing_video_id not in valid_ytids:
#             existing_video_ids.remove(existing_video_id)
#             os.remove(os.path.join(input_videos_folder, existing_video_id + ".mp4"))
#             print(f"Removed video {existing_video_id}.mp4 as it's not in ytids.txt")

# print(f"Found {len(existing_video_ids)} existing videos")

# youtube_ids_all = set()
# total_hours_all = 0.0

# # ------------------------------------------------
# # 2. filter CSVs, count hours, regenerate ytids
# # ------------------------------------------------
# for name in csv_names:
#     split = name.split("_")[0]

#     input_csv_path = os.path.join(input_csv_folder, name)
#     output_csv_path = os.path.join(output_folder, name)

#     df = pd.read_csv(input_csv_path)

#     # extract youtube_id if needed
#     if "youtube_id" not in df.columns:
#         df["youtube_id"] = df["file_path"].str.split("/", n=1).str[0]

#     # keep only rows whose video exists
#     df_filtered = df[df["youtube_id"].isin(existing_video_ids)].reset_index(drop=True)

#     # ---- count hours ----
#     total_frames = df_filtered["length"].sum()
#     hours = total_frames / FPS / 3600
#     total_hours_all += hours

#     print(f"{name}: {hours:.2f} hours")

#     # save filtered CSV
#     df_filtered.to_csv(output_csv_path, index=False)

#     # write split ytids
#     kept_ids = sorted(df_filtered["youtube_id"].unique())
#     youtube_ids_all.update(kept_ids)

    

#     ytids_path = os.path.join(output_folder, f"ytids_{split}.txt")
#     with open(ytids_path, "w") as f:
#         for yt_id in kept_ids:
#             f.write(f"{yt_id}\n")

#     print(
#         f"  kept {len(df_filtered)} / {len(df)} segments "
#         f"({len(kept_ids)} videos)"
#     )

# # ------------------------------------------------
# # 3. write global ytids + total hours
# # ------------------------------------------------
# all_ytids_path = os.path.join(output_folder, "ytids.txt")
# with open(all_ytids_path, "w") as f:
#     for yt_id in sorted(youtube_ids_all):
#         f.write(f"{yt_id}\n")

# print(f"Wrote {len(youtube_ids_all)} unique YouTube IDs")

# print(f"Total hours (train + val): {total_hours_all:.2f} hours")


Found 4168 existing videos


train_major.csv: 294.92 hours
  kept 316179 / 321340 segments (4136 videos)
val_major.csv: 2.73 hours
  kept 2976 / 3037 segments (32 videos)
Wrote 4168 unique YouTube IDs
Total hours (train + val): 297.65 hours


In [ ]:
# do this before preprocess
import os
import shutil
from tqdm import tqdm
base_folder = "/mnt/sdb/yuran/av_hubert/datasets/multivsr"
video_root = os.path.join(base_folder, "videos")

existing_video_list = [f.split(".")[0] for f in os.listdir(video_root) if f.endswith(".mp4")]
existing_video_list = set(existing_video_list)
face_folder = os.path.join(base_folder, "multivsr")
# remove any subfolder of face_folder that does NOT correspond to an existing video
for subfolder in tqdm(os.listdir(face_folder),total=len(os.listdir(face_folder))):
    subfolder_path = os.path.join(face_folder, subfolder)
    if os.path.isdir(subfolder_path) and subfolder not in existing_video_list:
        shutil.rmtree(subfolder_path)
        
surviving_folders = [f for f in os.listdir(face_folder) if os.path.isdir(os.path.join(face_folder, f))]
print(f"After cleanup, {len(surviving_folders)} face folders remain.")


  1%|          | 798/125245 [00:00<01:32, 1351.37it/s]

100%|██████████| 125245/125245 [02:04<00:00, 1006.08it/s]

After cleanup, 4165 face folders remain.


In [ ]:
# import os
# from moviepy.editor import VideoFileClip
# from tqdm import tqdm

# video_root = "/mnt/sdb/yuran/av_hubert/datasets/multivsr/multivsr"

# total_seconds = 0

# # Read train/val IDs
# train_ids_file = '/mnt/sdb/yuran/av_hubert/datasets/multivsr/final_metadata/ytids_train.txt'
# with open(train_ids_file, 'r') as f:
#     train_ids_set = set(line.strip() for line in f)

# val_ids_file = '/mnt/sdb/yuran/av_hubert/datasets/multivsr/final_metadata/ytids_val.txt'
# with open(val_ids_file, 'r') as f:
#     val_ids_set = set(line.strip() for line in f)

# train_length = 0
# val_length = 0
# train_count = 0
# val_count = 0

# # Walk through the video directory  
# for root, dirs, files in tqdm(os.walk(video_root),total=len(os.listdir(video_root)) ):
#     for file in files:
#         if file.lower().endswith(".mp4"):
#             video_path = os.path.join(root, file)
#             # Video ID is assumed to be the parent folder name
#             video_id = os.path.basename(root)
#             try:
#                 clip = VideoFileClip(video_path)
#                 duration = clip.duration
#                 total_seconds += duration
#                 clip.close()
#                 if video_id in train_ids_set:
#                     train_length += duration
#                     train_count+=1
#                 else:
#                     val_length += duration
#                     val_count+=1
#             except Exception as e:
#                 print(f"Failed to read {video_path}: {e}")
#                 continue  # skip this file if failed

            
# print(f"Total duration: {total_seconds/3600:.2f} hours")
# print(f"Train duration: {train_length/3600:.2f} hours over {train_count} videos")
# print(f"Validation duration: {val_length/3600:.2f} hours over {val_count} videos")

444it [34:33,  4.67s/it]                         

Total duration: 39.29 hours
Train duration: 35.99 hours over 28614 videos
Validation duration: 3.30 hours over 2535 videos


In [ ]:
# existing_video_dir="/mnt/sdb/yuran/av_hubert/datasets/multivsr/videos"
# import json
# import os
# existing_videos_list=os.listdir(existing_video_dir)
# # sort
# existing_videos_list.sort()
# with open("exist_ids.txt", "w") as f:
#     for vid in existing_videos_list:
#         f.write(vid.replace(".mp4","") + "\n")